# LANL 01 Preprocess And Segment

Build the LANL proxy state, detect failure resets, export a selected cycle, and save the segmentation summary.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.config import LANL_CONFIG
from src.io.lanl import load_lanl_train, scan_lanl_reset_indices, sample_lanl_cycle
from src.preprocess.lanl import clean_lanl_dataframe, add_lanl_proxies
from src.segmentation.lanl import segment_lanl_cycles
from src.segmentation.common import summarize_segments
from src.utils.plotting import plot_signal_panel, plot_segment_boundaries

RESULTS_DIR = LANL_CONFIG.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
INITIAL_ROWS = 1_000_000
RESET_BUFFER_ROWS = 10_000
SELECTED_CYCLE_MAX_ROWS = 100_000


In [2]:
reset_indices = scan_lanl_reset_indices(count=2)
if len(reset_indices) < 2:
    raise RuntimeError("LANL segmentation requires at least two resets to export a full cycle.")

resolved_rows = max(INITIAL_ROWS, reset_indices[0] + RESET_BUFFER_ROWS)
subset_df = add_lanl_proxies(clean_lanl_dataframe(load_lanl_train(nrows=resolved_rows)))
segments = segment_lanl_cycles(
    subset_df,
    failure_col="time_to_failure",
    min_cycle_length=LANL_CONFIG.segmentation["min_cycle_length"],
)
segment_summary = summarize_segments(segments)

cycle_df, cycle_step = sample_lanl_cycle(
    reset_indices[0],
    reset_indices[1],
    max_rows=SELECTED_CYCLE_MAX_ROWS,
)
cycle_df = add_lanl_proxies(clean_lanl_dataframe(cycle_df))

cycle_path = RESULTS_DIR / "lanl_selected_cycle.csv"
metadata_path = RESULTS_DIR / "lanl_selected_cycle_metadata.json"
segmentation_path = RESULTS_DIR / "segmentation_summary.json"

cycle_df.to_csv(cycle_path, index=False)
metadata = {
    "dataset": "lanl",
    "state_columns": ["tau_proxy", "V_proxy"],
    "state_labels": ["tau_proxy", "V_proxy"],
    "velocity_mode": "derived_proxy",
    "cycle_step": int(cycle_step),
    "cycle_path": str(cycle_path),
    "selected_cycle_range": [int(reset_indices[0]), int(reset_indices[1])],
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
segmentation_path.write_text(
    json.dumps(
        {
            "reset_indices": reset_indices,
            "resolved_subset_rows": int(resolved_rows),
            "segments": segments,
            "segment_summary": segment_summary,
            "selected_cycle_path": str(cycle_path),
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("Detected reset indices:", reset_indices)
print("Resolved subset rows:", resolved_rows)
print("Segment summary:", segment_summary)
print("Saved selected cycle to:", cycle_path)


Detected reset indices: [5656574, 50085878]
Resolved subset rows: 5666574
Segment summary: {'count': 1, 'lengths': [5656574], 'mean_length': 5656574.0, 'min_length': 5656574, 'max_length': 5656574}
Saved selected cycle to: C:\Users\carla\Desktop\EECE 798K\Project\results\lanl\lanl_selected_cycle.csv


In [3]:
reset_window = subset_df.iloc[max(0, reset_indices[0] - 20_000): reset_indices[0] + 20_000].copy()
plot_signal_panel(reset_window, "time", ["acoustic_data", "tau_proxy", "time_to_failure"], "LANL reset window")
plot_segment_boundaries(subset_df.iloc[: min(len(subset_df), reset_indices[0] + 10_000)], "time", "time_to_failure", segments, "LANL segmentation boundaries")
